# <font color="#418FDE" size="6.5" uppercase>**Klassifikation mit NumPy**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Berechnen Klassifikationsscores, Wahrscheinlichkeiten und binäre Verluste. 
- Trainieren eine kleine logistische Regression mit NumPy. 
- Vergleichen logistische Regression, k-NN, Naive Bayes und Baselines. 


## **1. Lineare Klassifikation**

### **1.1. Perzeptron Schwelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_01_01.jpg?v=1787639971" width="250">



>* Merkmale werden gewichtet und zum Score kombiniert
>* Die Schwelle entscheidet zwischen zwei Klassen

>* Perzeptron trennt Klassen durch lineare Grenzen
>* Schwelle steuert Fehlalarme und übersehene Fälle

>* Klare Entscheidungen durch feste Score-Schwellen
>* Begrenzt bei Unsicherheit und nichtlinearen Klassen



In [ ]:
#@title Python-Code - Perzeptron Schwelle

# Dieses Beispiel zeigt eine einfache Perzeptron-Schwelle.
# Scores entstehen aus Merkmalen, Gewichten und Bias.
# Die Grafik zeigt Entscheidungen bei verschiedenen Schwellen.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten beschreiben E-Mails mit zwei Merkmalen.
features = np.array([[1, 0], [2, 1], [1, 3], [3, 2], [4, 1], [2, 4]])
labels = np.array([0, 0, 1, 1, 1, 1])

# Gewichte legen fest, wie stark jedes Merkmal zählt.
weights = np.array([0.8, 1.2])
bias = -2.5
scores = features @ weights + bias

# Eine Schwelle verwandelt Scores in binäre Entscheidungen.
threshold = 0.0
predictions = (scores >= threshold).astype(int)
accuracy = np.mean(predictions == labels)

# Eine zweite Schwelle zeigt strengere positive Entscheidungen.
higher_threshold = 1.5
strict_predictions = (scores >= higher_threshold).astype(int)
strict_positives = int(np.sum(strict_predictions))

print("Perzeptron-Schwelle mit zwei E-Mail-Merkmalen")
print(f"Gewichte: Links={weights[0]:.1f}, Spam-Wörter={weights[1]:.1f}")
print(f"Bias: {bias:.1f}, Schwelle: {threshold:.1f}")
print(f"Scores: {np.round(scores, 2).tolist()}")
print(f"Vorhersagen: {predictions.tolist()}, Genauigkeit: {accuracy:.2f}")
print(f"Bei Schwelle {higher_threshold:.1f}: {strict_positives} positive Fälle")

# Die Entscheidungsgrenze erfüllt score gleich Schwelle.
x_values = np.linspace(0, 5, 100)
y_values = (threshold - bias - weights[0] * x_values) / weights[1]

# Die Punkte zeigen Daten, die Linie zeigt die Schwelle.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(features[:, 0], features[:, 1], c=labels, cmap="coolwarm", s=90)

ax.plot(x_values, y_values, color="black", label="Schwelle: Score = 0")
ax.set_title("Perzeptron: Score und Schwelle")
ax.set_xlabel("Anzahl Links")
ax.set_ylabel("Anzahl Spam-Wörter")

ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.legend()
plt.show()



### **1.2. Sigmoid und Logits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_01_02.jpg?v=1787639973" width="250">



>* Lineare Scores zeigen Richtung und Stärke
>* Sigmoid macht daraus Wahrscheinlichkeiten zwischen null und eins

>* Logit ist rohe Evidenz vor Sigmoid
>* Nahe der Grenze ändern Wahrscheinlichkeiten stark

>* Logits rechnen, Wahrscheinlichkeiten verständlich interpretieren
>* Modellwahrscheinlichkeiten sind nicht automatisch perfekt kalibriert



In [ ]:
#@title Python-Code - Sigmoid und Logits

# Dieses Beispiel zeigt Logits und Sigmoid-Wahrscheinlichkeiten.
# Wir berechnen Scores für einfache Klassifikationsfälle.
# Die Grafik macht die Entscheidungsgrenze sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Diese Funktion wandelt jeden Logit in eine Wahrscheinlichkeit um.
def sigmoid(logits):
    return 1.0 / (1.0 + np.exp(-logits))

# Kleine Beispiel-Logits zeigen negative, neutrale und positive Evidenz.
example_logits = np.array([-4.0, -1.0, 0.0, 1.0, 4.0])
example_probabilities = sigmoid(example_logits)

# Diese Prüfung schützt vor unerwarteten Formfehlern.
if example_logits.shape != example_probabilities.shape:
    raise ValueError("Logits und Wahrscheinlichkeiten müssen gleich lang sein.")

print("Logit -> Wahrscheinlichkeit für Klasse 1")
for logit, probability in zip(example_logits, example_probabilities):
    print(f"{logit:>5.1f} -> {probability:.3f}")

# Eine glatte Kurve zeigt den Zusammenhang über viele Logits.
logit_grid = np.linspace(-8.0, 8.0, 300)
probability_grid = sigmoid(logit_grid)

# Die Entscheidungsgrenze liegt bei Logit null und Wahrscheinlichkeit 0,5.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(logit_grid, probability_grid, label="Sigmoid")
ax.axvline(0.0, color="gray", linestyle="--", label="Logit = 0")

ax.axhline(0.5, color="orange", linestyle=":", label="Wahrscheinlichkeit = 0,5")
ax.scatter(example_logits, example_probabilities, color="red", zorder=3)
ax.set_title("Sigmoid: vom Logit zur Wahrscheinlichkeit")
ax.set_xlabel("Logit, also linearer Score")

ax.set_ylabel("Wahrscheinlichkeit für Klasse 1")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.show()



### **1.3. Kreuzentropie verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_01_03.jpg?v=1787639975" width="250">



>* Kreuzentropie bewertet vorhergesagte Wahrscheinlichkeiten.
>* Sichere falsche Vorhersagen werden stark bestraft.

>* Kreuzentropie misst durchschnittliche Modellüberraschung
>* Gute Wahrscheinlichkeiten zählen trotz gleicher Trefferzahl

>* Logits werden zu Wahrscheinlichkeiten umgewandelt
>* Kreuzentropie bestraft falsche Sicherheit stark



In [ ]:
#@title Python-Code - Kreuzentropie verstehen

# Dieses Beispiel macht Kreuzentropie anschaulich.
# Wahrscheinlichkeiten werden mit echten Klassen verglichen.
# Sichere Fehler erhalten besonders große Verluste.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Daten zeigen vier binäre Vorhersagen.
true_labels = np.array([1, 1, 0, 0])
predicted_probs = np.array([0.90, 0.55, 0.40, 0.02])

# Diese Prüfung verhindert unpassende Wahrscheinlichkeiten.
if np.any((predicted_probs <= 0) | (predicted_probs >= 1)):
    raise ValueError("Wahrscheinlichkeiten müssen zwischen null und eins liegen.")

# Kreuzentropie betrachtet die Wahrscheinlichkeit der richtigen Klasse.
correct_class_probs = np.where(true_labels == 1, predicted_probs, 1 - predicted_probs)
losses = -np.log(correct_class_probs)

# Der Mittelwert ist der gesamte Kreuzentropie-Verlust.
mean_loss = np.mean(losses)
print(f"Wahre Klassen: {true_labels.tolist()}")
print(f"Vorhergesagte Wahrscheinlichkeiten: {predicted_probs.round(2).tolist()}")
print(f"Verlust pro Beispiel: {losses.round(3).tolist()}")
print(f"Mittlere Kreuzentropie: {mean_loss:.3f}")

# Diese Kurve zeigt die Strafe für sichere Fehler.
prob_grid = np.linspace(0.01, 0.99, 99)
loss_for_positive = -np.log(prob_grid)

# Ein positives Beispiel leidet bei niedriger Wahrscheinlichkeit stark.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(prob_grid, loss_for_positive, color="tab:red", label="wahre Klasse = 1")
ax.scatter(predicted_probs[:2], losses[:2], color="black", zorder=3)

# Achsen und Titel machen die Bedeutung der Kurve klar.
ax.set_title("Kreuzentropie für ein positives Beispiel")
ax.set_xlabel("Vorhergesagte Wahrscheinlichkeit für Klasse 1")
ax.set_ylabel("Kreuzentropie-Verlust")
ax.legend()
plt.show()



## **2. Logistische Regression trainieren**

### **2.1. Gradienten berechnen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_02_01.jpg?v=1787639983" width="250">



>* Gradienten zeigen Richtung und Stärke der Fehleränderung
>* Gewichte werden angepasst, um Verlust zu senken

>* NumPy berechnet Gradienten vektorisiert für viele Beispiele.
>* Gemittelte Fehler liefern stabile Gewichtsanpassungen.

>* Korrekte Gradienten ermöglichen stabiles Lernen
>* Fehler führen zu gezielten Parameteranpassungen



In [ ]:
#@title Python-Code - Gradienten berechnen

# Wir berechnen Gradienten für logistische Regression.
# Kleine Daten zeigen Fehler und Korrekturrichtung.
# Der Plot zeigt den sinkenden Verlust.

import numpy as np
import matplotlib.pyplot as plt

# Diese Mini-Daten haben zwei Merkmale und binäre Labels.
features = np.array([[0.2, 1.0], [1.0, 0.3], [1.2, 1.1], [2.0, 1.5]])
labels = np.array([0.0, 0.0, 1.0, 1.0])

# Die Formprüfung verhindert unpassende Matrixoperationen.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Jede Zeile braucht genau ein Label.")

# Startwerte sind absichtlich einfach und nachvollziehbar.
weights = np.array([0.0, 0.0])
bias = 0.0
learning_rate = 0.8
loss_history = []

# Die Sigmoid-Funktion wandelt Scores in Wahrscheinlichkeiten um.
def sigmoid(scores):
    return 1.0 / (1.0 + np.exp(-scores))

# Mehrere kleine Schritte zeigen die Wirkung der Gradienten.
for step in range(12):
    scores = features @ weights + bias
    probabilities = sigmoid(scores)
    errors = probabilities - labels

    weight_gradient = features.T @ errors / labels.size
    bias_gradient = np.mean(errors)
    weights = weights - learning_rate * weight_gradient
    bias = bias - learning_rate * bias_gradient

    clipped = np.clip(probabilities, 1e-9, 1.0 - 1e-9)
    loss = -np.mean(labels * np.log(clipped) + (1.0 - labels) * np.log(1.0 - clipped))
    loss_history.append(loss)

# Die ersten Gradienten erklären die Richtung der Anpassung.
start_probabilities = np.full(labels.shape, 0.5)
start_errors = start_probabilities - labels
start_gradient = features.T @ start_errors / labels.size

print("Start-Gradient Gewichte:", np.round(start_gradient, 3))
print("Gelernte Gewichte:", np.round(weights, 3))
print("Gelernter Bias:", round(float(bias), 3))
print("Verlust vorher/nachher:", round(loss_history[0], 3), "->", round(loss_history[-1], 3))

# Der Verlauf macht sichtbar, dass Gradienten den Verlust senken.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
ax.set_title("Verlust sinkt durch Gradientenabstieg")
ax.set_xlabel("Trainingsschritt")
ax.set_ylabel("Logistischer Verlust")
ax.grid(True, alpha=0.3)
plt.show()



### **2.2. Modell iterativ trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_02_02.jpg?v=1787639987" width="250">



>* Modell verbessert Parameter Schritt für Schritt
>* NumPy macht den Optimierungsprozess sichtbar

>* Lernrate steuert die Schrittgröße beim Training
>* Verlustverlauf zeigt nötige Anpassungen

>* Lernverlauf zeigt, wie Merkmale wichtiger werden
>* Zurückgehaltene Daten prüfen echte Verallgemeinerung



In [ ]:
#@title Python-Code - Modell iterativ trainieren

# Wir trainieren logistische Regression Schritt für Schritt.
# Der Verlust zeigt den Lernfortschritt sichtbar.
# Am Ende klassifiziert das Modell neue Punkte.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Ein kleiner Datensatz macht die Rechenschritte übersichtlich.
features, labels = make_classification(
    n_samples=160, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.4, random_state=42
)

# Die Aufteilung prüft später echte Vorhersagen.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Eine einfache Prüfung verhindert unklare Formfehler.
if X_train_scaled.shape[1] != 2:
    raise ValueError("Dieser Kursabschnitt erwartet genau zwei Merkmale.")

# Startwerte nahe null bedeuten anfangs unsichere Vorhersagen.
weights = np.zeros(X_train_scaled.shape[1])
bias = 0.0
learning_rate = 0.25
iterations = 80

# Diese Liste speichert den Verlust jeder Iteration.
loss_history = []
sample_count = X_train_scaled.shape[0]
epsilon = 1e-12

# Jede Runde berechnet Vorhersagen, Gradienten und Updates.
for step in range(iterations):
    scores = X_train_scaled @ weights + bias
    probabilities = 1.0 / (1.0 + np.exp(-scores))
    errors = probabilities - y_train

    loss = -np.mean(
        y_train * np.log(probabilities + epsilon)
        + (1 - y_train) * np.log(1 - probabilities + epsilon)
    )
    loss_history.append(loss)

    weight_gradient = (X_train_scaled.T @ errors) / sample_count
    bias_gradient = np.mean(errors)
    weights = weights - learning_rate * weight_gradient
    bias = bias - learning_rate * bias_gradient

# Nach dem Training werden Testwahrscheinlichkeiten klassifiziert.
test_scores = X_test_scaled @ weights + bias
test_probabilities = 1.0 / (1.0 + np.exp(-test_scores))
test_predictions = (test_probabilities >= 0.5).astype(int)

# Kurze Kennzahlen verbinden Optimierung und Klassifikation.
accuracy = accuracy_score(y_test, test_predictions)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Startverlust: {loss_history[0]:.3f}")
print(f"Endverlust: {loss_history[-1]:.3f}")
print(f"Testgenauigkeit: {accuracy:.3f}")

# Die Kurve zeigt, ob die Lernrate sinnvoll arbeitet.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, iterations + 1), loss_history, label="Trainingsverlust")
ax.set_title("Iteratives Training einer logistischen Regression")

ax.set_xlabel("Iteration")
ax.set_ylabel("Binärer Kreuzentropie-Verlust")
ax.legend()
plt.show()



### **2.3. Entscheidungsgrenze verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_02_03.jpg?v=1787639985" width="250">



>* Grenze markiert den Wechsel zwischen Klassen
>* Sie entsteht aus gelernten Modellparametern

>* Lineare Grenze trennt Klassen verständlich
>* Gewichte zeigen Einfluss einzelner Merkmale

>* Lineare Grenzen passen nicht immer gut
>* Plausibilität mit Fehleranalysen und Baselines prüfen



In [ ]:
#@title Python-Code - Entscheidungsgrenze verstehen

# Wir visualisieren eine gelernte Entscheidungsgrenze.
# Logistische Regression trennt Punkte mit einer Geraden.
# Die Grafik zeigt Klassen, Wahrscheinlichkeiten und Grenze.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# Wir erzeugen kleine zweidimensionale Trainingsdaten.
features, labels = make_classification(
    n_samples=120,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.4,
    random_state=42,
)

# Diese Prüfung macht die erwartete Datenform sichtbar.
if features.shape != (120, 2):
    raise ValueError("Die Daten sollten 120 Zeilen und 2 Merkmale haben.")

# Das Modell lernt Gewichte und einen Achsenabschnitt.
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(features, labels)

# Die Grenze liegt dort, wo die Wahrscheinlichkeit genau 0,5 ist.
weight_x = model.coef_[0, 0]
weight_y = model.coef_[0, 1]
intercept = model.intercept_[0]

# Wir berechnen Punkte der Geraden aus den gelernten Parametern.
x_values = np.linspace(features[:, 0].min() - 0.5, features[:, 0].max() + 0.5, 100)
y_values = -(weight_x * x_values + intercept) / weight_y

# Drei Beispielpunkte zeigen den Zusammenhang von Score und Klasse.
example_points = np.array([[-2.0, -1.0], [0.0, 0.0], [2.0, 1.0]])
example_probabilities = model.predict_proba(example_points)[:, 1]
example_predictions = model.predict(example_points)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Gewicht Merkmal 1: {weight_x:.2f}")
print(f"Gewicht Merkmal 2: {weight_y:.2f}")
print(f"Achsenabschnitt: {intercept:.2f}")
print("Beispiel: Wahrscheinlichkeit -> vorhergesagte Klasse")

for probability, prediction in zip(example_probabilities, example_predictions):
    print(f"{probability:.2f} -> {prediction}")

# Die Grafik zeigt Datenpunkte und gelernte Entscheidungsgrenze.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=labels,
    cmap="coolwarm",
    edgecolor="black",
    alpha=0.8,
)

ax.plot(x_values, y_values, color="black", linewidth=2, label="Grenze bei p = 0,5")
ax.set_title("Entscheidungsgrenze einer logistischen Regression")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



## **3. Einfache Modellvergleiche**

### **3.1. k nächste Nachbarn**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_03_01.jpg?v=1787639977" width="250">



>* Neue Punkte folgen ähnlichen bekannten Beispielen
>* Trainingsdaten dienen als Gedächtnis

>* k bestimmt Empfindlichkeit und Glättung.
>* Skalierung prägt, welche Punkte ähnlich wirken.

>* k-NN ist anschaulich, aber vorhersageaufwendig
>* Gute Ähnlichkeit schlägt einfache Baselines



In [ ]:
#@title Python-Code - k nächste Nachbarn

# Wir vergleichen k-NN mit einer einfachen Baseline.
# Nähe entscheidet über die vorhergesagte Klasse.
# Skalierung macht den Vergleich fairer und stabiler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Wir nutzen zwei Iris-Merkmale für eine anschauliche Grafik.
iris = load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Der Skalierer lernt nur aus den Trainingsdaten.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# k-NN speichert Beispiele und stimmt lokal ab.
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Die Baseline ignoriert Merkmale vollständig.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)

# Wir vergleichen beide Modelle auf denselben Testdaten.
knn_accuracy = accuracy_score(y_test, knn.predict(X_test_scaled))
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test_scaled))

# Ein einzelner Testpunkt zeigt die Nachbarschaftsidee konkret.
example_index = 0
example_point = X_test_scaled[example_index:example_index + 1]
distances, neighbor_indices = knn.kneighbors(example_point)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"k-NN Genauigkeit: {knn_accuracy:.2f}")
print(f"Baseline Genauigkeit: {baseline_accuracy:.2f}")
print(f"Nachbarklassen für einen Testpunkt: {y_train[neighbor_indices[0]].tolist()}")

# Die Grafik zeigt Trainingspunkte, Testpunkt und Nachbarn.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap="viridis", label="Training"
)

ax.scatter(
    example_point[:, 0], example_point[:, 1], c="red", s=120, marker="X", label="Testpunkt"
)
ax.scatter(
    X_train_scaled[neighbor_indices[0], 0], X_train_scaled[neighbor_indices[0], 1],
    facecolors="none", edgecolors="black", s=180, label="5 nächste Nachbarn"
)

ax.set_title("k-NN: Entscheidung durch nahe Trainingsbeispiele")
ax.set_xlabel("Blütenblattlänge, skaliert")
ax.set_ylabel("Blütenblattbreite, skaliert")
ax.legend(loc="best")
plt.show()



### **3.2. Naive Bayes verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_03_02.jpg?v=1787639979" width="250">



>* Wählt die plausibelste Klasse anhand von Merkmalen
>* Nutzt naive Unabhängigkeit für stabile Schätzungen

>* Spam-Erkennung durch typische Wörter
>* Schnell, sparsam und gut bei vielen Merkmalen

>* Naive Bayes ist schnell, aber vereinfacht.
>* Vergleiche Modelle fair mit gleichen Daten.



In [ ]:
#@title Python-Code - Naive Bayes verstehen

# Dieses Beispiel zeigt Naive Bayes mit Wortzählungen.
# Wir vergleichen Klassen über einfache bedingte Wahrscheinlichkeiten.
# Am Ende sehen wir eine nachvollziehbare Spamentscheidung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.naive_bayes import MultinomialNB

# Kleine Trainingsdaten zählen drei typische Wörter.
feature_names = np.array(["gewinn", "projekt", "dringend"])
X_train = np.array([[3, 0, 1], [2, 0, 2], [0, 3, 0], [0, 2, 1]])
y_train = np.array([1, 1, 0, 0])

# Die neue Nachricht enthält Gewinn und Dringlichkeit.
new_message = np.array([[1, 0, 1]])

# MultinomialNB passt gut zu gezählten Wörtern.
model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

# Wir berechnen Wahrscheinlichkeiten für beide Klassen.
class_probabilities = model.predict_proba(new_message)[0]
predicted_class = model.predict(new_message)[0]

# Diese Werte zeigen, welche Wörter pro Klasse typisch sind.
word_probabilities = np.exp(model.feature_log_prob_)
spam_word_probabilities = word_probabilities[1]
normal_word_probabilities = word_probabilities[0]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"P(normal): {class_probabilities[0]:.2f}")
print(f"P(spam): {class_probabilities[1]:.2f}")
print(f"Vorhersage: {'Spam' if predicted_class == 1 else 'Normal'}")

# Das Diagramm vergleicht Worttypik in beiden Klassen.
fig, ax = plt.subplots(figsize=(7, 4))
x_positions = np.arange(len(feature_names))
bar_width = 0.35

ax.bar(x_positions - bar_width / 2, normal_word_probabilities, bar_width, label="Normal")
ax.bar(x_positions + bar_width / 2, spam_word_probabilities, bar_width, label="Spam")
ax.set_title("Naive Bayes: Wortwahrscheinlichkeiten je Klasse")
ax.set_xlabel("Wort")

ax.set_ylabel("Geschätzte Wahrscheinlichkeit")
ax.set_xticks(x_positions)
ax.set_xticklabels(feature_names)
ax.legend()

plt.show()



### **3.3. Modelle fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_03_03.jpg?v=1787639981" width="250">



>* Gleiche Trainings-Test-Trennung für alle Modelle
>* Baselines zeigen den nötigen Mindestnutzen

>* Passende Metriken statt nur Genauigkeit
>* Schwellenwerte an Anwendung und Risiken anpassen

>* Leistung, Stabilität und Aufwand gemeinsam bewerten
>* Praktische Anforderungen entscheiden die Modellwahl



In [ ]:
#@title Python-Code - Modelle fair vergleichen

# Wir vergleichen Modelle unter gleichen Bedingungen.
# Fairness bedeutet gleiche Daten und Vorverarbeitung.
# Die Grafik zeigt Testgenauigkeit gegen Trainingszeit.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Beispielannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Alle Modelle erhalten exakt dieselbe Aufteilung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Skalierung wird nur im Trainingsanteil gelernt.
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistische Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
}

# Wir messen dieselbe Metrik für jedes Modell.
names = []
accuracies = []
fit_times = []

# Die Schleife trainiert jedes Modell genau einmal.
for name, model in models.items():
    start = np.datetime64("now", "ms")
    model.fit(X_train, y_train)
    end = np.datetime64("now", "ms")

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    elapsed_ms = (end - start) / np.timedelta64(1, "ms")

    names.append(name)
    accuracies.append(accuracy)
    fit_times.append(float(elapsed_ms))

# Kurze Ausgabe zeigt Version und wichtigste Ergebnisse.
print(f"scikit-learn-Version: {sklearn.__version__}")
print("Fairer Vergleich: gleiche Aufteilung, gleiche Testdaten.")

# Nur drei Ergebniszeilen bleiben übersichtlich.
for name, accuracy in zip(names[:3], accuracies[:3]):
    print(f"{name}: Genauigkeit {accuracy:.3f}")

# Ein Balkendiagramm macht die Modellunterschiede sichtbar.
fig, ax = plt.subplots(figsize=(8, 4))
bar_positions = np.arange(len(names))
ax.bar(bar_positions, accuracies, color="steelblue")

# Achsen und Titel erklären die faire Vergleichsgröße.
ax.set_xticks(bar_positions)
ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("Testgenauigkeit")
ax.set_xlabel("Modell")

# Die y-Achse fokussiert auf sinnvolle Genauigkeitswerte.
ax.set_ylim(0.8, 1.0)
ax.set_title("Modelle fair vergleichen: gleiche Testdaten")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Klassifikation mit NumPy**</font>


In this lecture, you learned to:
- Berechnen Klassifikationsscores, Wahrscheinlichkeiten und binäre Verluste. 
- Trainieren eine kleine logistische Regression mit NumPy. 
- Vergleichen logistische Regression, k-NN, Naive Bayes und Baselines. 

In the next Lecture (Lecture B), we will go over 'Cluster und PCA'